In [1]:
import numpy as np
import sys, platform
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian, Pauli
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator

print("Python full:", sys.version)          # e.g. '3.12.5 (main, ...)'
print("Python tuple:", sys.version_info)     # e.g. sys.version_info(major=3, minor=12, micro=5, ...)
print("Python short:", platform.python_version())  # e.g. '3.12.5'
print("Executable:", sys.executable)         # path to the kernel's python


Python full: 3.12.9 | packaged by conda-forge | (main, Mar  4 2025, 22:44:42) [Clang 18.1.8 ]
Python tuple: sys.version_info(major=3, minor=12, micro=9, releaselevel='final', serial=0)
Python short: 3.12.9
Executable: /opt/homebrew/anaconda3/envs/qiskit2x/bin/python


In [2]:
## Using Estimator

from qiskit import QuantumCircuit
from qiskit.circuit import QuantumRegister, ClassicalRegister, Parameter, ParameterVector
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2, SamplerV2

qr=QuantumRegister(3, 'myreg')
cr=ClassicalRegister(3, 'creg_a')
qc=QuantumCircuit(qr, cr)


### Exercise 1

In [3]:
def maxcut_stats(edges, counts):
    """
    edges: list of (u, v) undirected edges, nodes are 0-indexed ints
    counts: dict mapping bitstring -> int count, e.g. {"0101": 30, "0000": 10}
            bitstring convention: LEFTMOST char is node 0 (so s[0] is node 0)
    Returns: (expected_cut_value: float, best_cut_value_observed: int)
    """
    # check max node
    max_node = max(max(u,v) for (u,v) in edges)
    total_counts = sum(counts.values())
    exp_cut = 0.0
    best_cut = 0
    for s, cnt in counts.items():
        if len(s) != max_node + 1:
            raise ValueError("Bitstrings in counts have length {}, but edges have node {}".format(len(s), max_node))
        obj = maxcut_obj(edges, s)
        exp_cut += obj * (cnt / total_counts)
        if obj > best_cut:
            best_cut = obj
    return exp_cut, best_cut

def maxcut_obj(edges, s):
    """
    s: bitstring
    Returns: objective
    """
    obj = 0
    for u, v in edges:
        if s[u] != s[v]:
            obj += 1
    return obj
    

In [4]:
edges  = [(0,1),(1,2),(2,3),(3,0)]  # 4-cycle
counts = {"0000": 10, "0101": 30}
# Expected: expected_cut_value = 3.0, best_cut_value_observed = 4
maxcut_stats(edges, counts)

(3.0, 4)

## TTS Exercise

In [5]:
def tts_from_runs(runs, threshold, alpha=0.99):
    """
    runs: list of dicts, each like {"time_s": float, "obj": int}
    threshold: objective threshold for success
    Returns: (p_success: float, t_run: float, tts: float)
    """
    if alpha <= 0 or alpha >= 1:
        raise ValueError("alpha must be in (0, 1)")
    if not runs:
        raise ValueError("No runs provided")
    n_runs = len(runs)
    n_success = sum(1 for run in runs if run["obj"] >= threshold)
    p_success = n_success / n_runs
    t_run = np.mean([run["time_s"] for run in runs])
    if n_success == 0:
        tts = float('inf')
    elif n_success == n_runs:
        tts = t_run
    else:
        tts = t_run * np.ceil(np.log(1 - alpha) / np.log(1 - p_success))
    return p_success, t_run, tts


In [6]:
tts_from_runs([{"time_s": 1.0, "obj": 3}, {"time_s": 1.5, "obj": 4}, {"time_s": 0.5, "obj": 2}], threshold=4)

(0.3333333333333333, np.float64(1.0), np.float64(12.0))

In [7]:
def crossover_n(bench, alpha=0.99):
    """
    bench: list of entries, each like:
      {"n": int, "threshold": int,
       "classical": [runs...],
       "hybrid":    [runs...]}
    Returns: the smallest n where TTS_hybrid < TTS_classical, else None
    """
    for entry in bench:
        n = entry["n"]
        threshold = entry["threshold"]
        p_c, t_c, tts_c = tts_from_runs(entry["classical"], threshold, alpha)
        p_h, t_h, tts_h = tts_from_runs(entry["hybrid"], threshold, alpha)
        if tts_h < tts_c:
            return n
    return None

In [8]:
bench = [
  {"n": 8,  "threshold": 11,
   "classical": [{"time_s": 0.05, "obj": o} for o in [11,11,11,11,11,11,11,10,10,10]],
   "hybrid":    [{"time_s": 0.20, "obj": o} for o in [11,11,11,11,11,11,11,11,11,10]],
  },
  {"n": 12, "threshold": 18,
   "classical": [{"time_s": 0.20, "obj": o} for o in [18,18,18,17,17,17,17,17,17,17]],
   "hybrid":    [{"time_s": 0.50, "obj": o} for o in [18,18,18,18,18,18,17,17,17,17]],
  },
  {"n": 16, "threshold": 25,
   "classical": [{"time_s": 0.60, "obj": o} for o in [25,24,24,24,24,24,24,24,24,24]],
   "hybrid":    [{"time_s": 1.00, "obj": o} for o in [25,25,25,25,24,24,24,24,24,24]],
  },
]


In [9]:
crossover_n(bench)

16

## Exercise: Bell state + Empirical

In [10]:
def bell_zz_expectation(shots=2000, seed=123):
    """
    Build a 2-qubit Bell state circuit, run it on a QASM simulator,
    and compute the empirical expectation value <Z ⊗ Z> from the returned counts.

    Return: (exp_zz: float, counts: dict[str, int])
    """
    from qiskit import QuantumCircuit
    from qiskit_aer.primitives import SamplerV2

    # Build Bell state circuit
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure_all()

    # Initialize Qiskit Runtime Service
    # Use Sampler to run the circuit
    sampler = SamplerV2(seed=seed)

    # transpile
    #pm = generate_preset_pass_manager(backend=my_backend, optimization_level=3)
    #cir_isa = pm.run(qc)
    PUB = (qc, None)
    job = sampler.run([PUB], shots=shots)
    result = job.result()
    #counts = result.quasi_dicts[0].binary_probabilities()
    counts = result[0].data.meas.get_counts()
    print(counts)
    # Compute expectation value <Z ⊗ Z>
    #exp_zz = 0.0
    #for bitstring, prob in counts.items():
    #    z0 = 1 if bitstring[1] == '0' else -1  # Qubit 0
    #    z1 = 1 if bitstring[0] == '0' else -1  # Qubit 1
    #    exp_zz += z0 * z1 * prob

    exp_zz = 0.0
    for bitstring, count in counts.items():
        z0 = 1 if bitstring[1] == '0' else -1  # Qubit 0
        z1 = 1 if bitstring[0] == '0' else -1  # Qubit 1
        exp_zz += z0 * z1 * (count / shots)

    return exp_zz, counts   

In [11]:
shots = 2000

"""
Build a 2-qubit Bell state circuit, run it on a QASM simulator,
and compute the empirical expectation value <Z ⊗ Z> from the returned counts.

Return: (exp_zz: float, counts: dict[str, int])
"""
from qiskit import QuantumCircuit
#from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_aer.primitives import SamplerV2

# Build Bell state circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Initialize Qiskit Runtime Service
#service = QiskitRuntimeService()
#my_backend = service.backends()[0]
# Use Sampler to run the circuit
sampler = SamplerV2(seed=123)

# transpile
pm = generate_preset_pass_manager(backend=my_backend, optimization_level=3)
cir_isa = pm.run(qc)
PUB = (cir_isa, [])
job = sampler.run([PUB], shots=shots)
result = job.result()
#counts = result.quasi_dicts[0].binary_probabilities()
counts = result[0].data.meas.get_counts()
print(counts)

NameError: name 'my_backend' is not defined

In [ ]:
# Sanity Check
bell_zz_expectation(shots=2000, seed=123)

{'00': 993, '11': 1007}


(1.0, {'00': 993, '11': 1007})

## Exercise: Expected energy from samples

In [ ]:
def qubo_energy(Q, x):
    """
    Q: dict[(i,j)] -> float for i<=j (upper triangle incl diagonal)
    x: iterable of 0/1 ints length n
    Returns: float = sum_{i<=j} Q[i,j] * x_i * x_j
    """
    energy = 0.0
    n = len(x)
    for i in range(n):
        for j in range(i, n):
            q_ij = Q.get((i, j), 0.0)
            energy += q_ij * x[i] * x[j]
    return energy

Q = {
    (0, 0): 1.0,
    (0, 1): 0.5,
    (1, 1): 2.0
}
x = [1, 0]
result=qubo_energy(Q, x)  # should 

{(0, 0): 1.0, (0, 1): 0.5, (1, 1): 2.0}


In [ ]:
# Initialize dictionary J and list h
J = {}
h = [0.0] * len(x)

# Helper function to compute entry
def compute_entry(i, j, value):
    return value/4  # Example computation, replace with actual logic

def compute_h(Q, i, j):
    value = -Q.get((i, j), 0.0)/2 - 0.25 * sum(Q.get((i, k), 0.0) for k in range(len(x)) if k != i)
    return value 

# Iterate over the upper diagonal entries of Q
for (i, j), value in Q.items():
    if i < j:  # Upper diagonal condition
        J[(i, j)] = compute_entry(i, j, value)
    elif i == j:
        h[i] = compute_h(Q, i, j)

offset = sum(Q.get((i, i), 0.0) for i in range(len(x))) / 2 + sum(Q.get((i, j), 0.0) for i in range(len(x)) for j in range(i+1, len(x))) / 4

In [ ]:
J

{(0, 1): 0.125}

In [ ]:
h

[-0.625, -1.0]

In [ ]:
offset

1.625

In [ ]:
def qubo_to_ising(Q, n):
    """
    Input: same Q as above, and number of variables n
    Output: (h, J, offset)
      h: list[float] length n   (linear terms)
      J: dict[(i,j)] -> float for i<j (couplers)
      offset: float

    Such that for all x in {0,1}^n with s=1-2x:
        qubo_energy(Q, x) == ising_energy(h, J, offset, s)
    """
    # Initialize dictionary J and list h
    J = {}
    h = [0.0] * len(x)
    
    # Helper function to compute entry
    def compute_entry(i, j, value):
        return value/4  # Example computation, replace with actual logic
    
    def compute_h(Q, i, j):
        value = -Q.get((i, j), 0.0)/2 - 0.25 * sum(Q.get((i, k), 0.0) for k in range(len(x)) if k != i)
        return value 
    
    # Iterate over the upper diagonal entries of Q
    for (i, j), value in Q.items():
        if i < j:  # Upper diagonal condition
            J[(i, j)] = compute_entry(i, j, value)
        elif i == j:
            h[i] = compute_h(Q, i, j)
    
    offset = sum(Q.get((i, i), 0.0) for i in range(len(x))) / 2 + sum(Q.get((i, j), 0.0) for i in range(len(x)) for j in range(i+1, len(x))) / 4
    return J, h, offset

## Exercise QAOA

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

def qaoa_maxcut_p1(n, edges):
    """
    n: number of qubits (nodes)
    edges: list of (u, v) or (u, v, w) where u,v are ints and w is a float weight (default 1.0)

    Returns:
      qc: QuantumCircuit with two Parameters: gamma, beta
          - initializes |+>^n
          - applies cost layer using ZZ interactions per edge
          - applies mixer layer using X rotations
    """
    gamma = Parameter('γ')
    beta = Parameter('β')

    # Create a quantum circuit with n qubits
    qc = QuantumCircuit(n)

    # Initialize all qubits in the |+> state
    qc.h(range(n))

    # Apply the cost layer
    for edge in edges:
        u, v = edge[:2]
        weight = edge[2] if len(edge) > 2 else 1.0
        qc.cx(u, v)
        qc.rz(2 * gamma * weight, v)
        qc.cx(u, v)

    # Apply the mixer layer
    for qubit in range(n):
        qc.rx(2 * beta, qubit)

    return qc, gamma, beta

In [ ]:
n = 3
edges = [(0,1), (1,2)]
qc = qaoa_maxcut_p1(n, edges)
print(qc)


(<qiskit.circuit.quantumcircuit.QuantumCircuit object at 0x117308e90>, Parameter(γ), Parameter(β))


## QAOA exercise

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

def qaoa_maxcut_p1(n, edges):
    """
    n: int, number of qubits/nodes
    edges: list of (u, v) or (u, v, w) where w is a float weight (default 1.0)

    Returns:
      qc: QuantumCircuit with exactly two Parameters: gamma and beta
          - prepares |+>^n
          - applies one cost layer using ZZ interactions for each edge
          - applies one mixer layer using RX rotations
          - measures all qubits
    Bitstring convention after measurement: leftmost bit is node 0.
    """
    gamma = Parameter('γ')
    beta = Parameter('β')

    # Create a quantum circuit with n qubits
    qc = QuantumCircuit(n)

    # Initialize all qubits in the |+> state
    qc.h(range(n))

    # Apply the cost layer
    for edge in edges:
        u, v = edge[:2]
        weight = edge[2] if len(edge) > 2 else 1.0
        qc.cx(u, v)
        qc.rz(2 * gamma * weight, v)
        qc.cx(u, v)

    # Apply the mixer layer
    for qubit in range(n):
        qc.rx(2 * beta, qubit)

    # Measure all qubits
    qc.measure_all()

    return qc, gamma, beta

In [2]:
qaoa_maxcut_p1(3, [(0,1), (1,2)])

(<qiskit.circuit.quantumcircuit.QuantumCircuit at 0x1231e58e0>,
 Parameter(γ),
 Parameter(β))

## Grid-search Qiskit for p=1

In [9]:
import numpy as np
gammas = np.linspace(0, np.pi, 9)      # 9 points
betas  = np.linspace(0, np.pi/2, 7)    # 7 points

n = 4
edges = [(0,1), (1,2), (2,3), (3,0)]  # 4-cycle, optimal cut = 4


In [10]:
from qiskit_aer.primitives import SamplerV2

In [18]:
def expected_cut_from_counts(counts, edges):
    """
    counts: dict[str,int]
    Returns: expected cut value (float)
    """
    total=0
    total_counts=sum([val for val in counts.values()])
    for k,v in counts.items():
        obj=maxcut_value(k, edges)
        total+=float(obj)*float(counts)
    return total/total_counts

In [19]:
def maxcut_value(bitstring, edges):
    """
    bitstring: str length n, LEFTMOST is node 0
    Returns: cut value (int)
    """
    total=0
    for e in edges:
        u,v=e
        if bitstring[u]!=bitstring[v]:
            total+=1
    return total

In [20]:
def qaoa_maxcut_p1(n, edges):
    """
    Returns a parameterized QuantumCircuit with Parameters gamma, beta:
      - H on all qubits
      - cost layer: ZZ interaction per edge with angle 2*gamma
      - mixer layer: RX(2*beta) on each qubit
      - measure_all()
    """
    for gamma in gammas:
        for beta in betas:
            qc = QuantumCircuit(n)
            qc.h(range(n))
            qc.rzz(2*gamma, range(n))
            qc.rx(2*beta, range(n))
            qc.measure_all()
            
            PUB=(qc, None)
            job=Sampler([PUB])
            result=job.results()
            counts=result[0].data.meas.get_counts()
            
            # get expected cut value
            expected_cut = expected_cut_from_counts(counts, edges)

            if expected_cut < best_obj:
                best_obj=expected_cut
                best_gamma=gamma
                best_beta=beta
                best_expected_cut=expected_cut
                
            return best_gamma, best_beta, expected_cut, best_bitstring_observed
                

SyntaxError: invalid syntax (3778275404.py, line 30)